# Telecom Customer Churn Analysis
## 05 — SQL Analysis

### Objective
Use SQL to answer business questions about customer churn and retention.

In [1]:
import pandas as pd
import sqlite3

In [5]:
# Load the cleaned dataset

dataset = r"C:\Users\HomePC\Desktop\Data Analysis Portfolio\telecom_customer_churn_analysis\data\telecom_customer_churn_clean.csv"
df_clean = pd.read_csv(dataset)

In [6]:
# Create a SQLite database connection

conn = sqlite3.connect("../data/telecom_churn.db")

In [8]:
# Load the data into a SQL table

df_clean.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Data loaded successfully!")

Data loaded successfully!


In [11]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_customers FROM customers;",
    conn
)

,total_customers
0,7043


In [19]:
# Question 1: How many customers are in each status?

query = """
SELECT
    "Customer Status",
    COUNT(*) AS customer_count
FROM customers
GROUP BY "Customer Status"
ORDER BY customer_count DESC;
"""

df_result = pd.read_sql_query(query, conn)

df_result

,Customer Status,customer_count
0,Stayed,4720
1,Churned,1869
2,Joined,454


In [29]:
# Alternate method

pd.read_sql_query(
    """SELECT
        "Customer Status",
        COUNT(*) AS customer_count
    FROM customers
    GROUP BY "Customer Status"
    ORDER BY customer_count DESC;""",
    conn
)

,Customer Status,customer_count
0,Stayed,4720
1,Churned,1869
2,Joined,454


In [22]:
# Question 2: What percentage of each contract group churned?

pd.read_sql_query(
    """
    SELECT
        Contract,
        COUNT(*) AS total_customers,
        SUM(
            CASE
                WHEN "Customer Status" = 'Churned' THEN 1
                ELSE 0
            END
        ) AS churned_customers,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN "Customer Status" = 'Churned' THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS churn_percentage
    FROM customers
    GROUP BY Contract
    ORDER BY churn_percentage DESC;    
    """,
    conn
)

,Contract,total_customers,churned_customers,churn_percentage
0,Month-to-Month,3610,1655,45.84
1,One Year,1550,166,10.71
2,Two Year,1883,48,2.55


In [23]:
# Question 3: Which internet types have the highest churn percentage?

pd.read_sql_query(
    """
SELECT
    "Internet Type",
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN "Customer Status" = 'Churned' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Customer Status" = 'Churned' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_percentage
    FROM customers
    WHERE "Internet Service" = 'Yes'
    GROUP BY "Internet Type"
    ORDER BY churn_percentage DESC;
    """,
    conn
)

,Internet Type,total_customers,churned_customers,churn_percentage
0,Fiber Optic,3035,1236,40.72
1,Cable,830,213,25.66
2,DSL,1652,307,18.58


In [26]:
# Contract churn analysis

pd.read_sql_query(
    """
    SELECT
        Contract,
        COUNT(*) AS total_customers,
        SUM(
            CASE
                WHEN "Customer Status" = 'Churned' THEN 1
                ELSE 0
            END
        ) AS churned_customers,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN "Customer Status" = 'Churned' THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS churn_percentage
    FROM customers
    GROUP BY Contract
    ORDER BY churn_percentage DESC;
    """,
    conn
)

,Contract,total_customers,churned_customers,churn_percentage
0,Month-to-Month,3610,1655,45.84
1,One Year,1550,166,10.71
2,Two Year,1883,48,2.55


### SQL Finding: Churn by Contract Type

The SQL analysis shows substantial differences in observed churn percentages across contract types. Month-to-month customers had the highest churn percentage at 45.84%, followed by one-year customers at 10.71%. Two-year customers had the lowest churn percentage at 2.55%.

**Business implication:** The company should investigate why month-to-month customers are more likely to churn. Potential areas to examine include pricing, customer tenure, service experience, and the reasons customers give for leaving.

**Conclusion:** Contract type is strongly associated with customer churn in this dataset. Further analysis is needed before concluding that longer contracts directly improve retention.

In [27]:
# Churn by customer tenure

pd.read_sql_query(
    """
SELECT
    CASE
        WHEN "Tenure in Months" <= 12 THEN '0-12 months'
        WHEN "Tenure in Months" <= 24 THEN '13-24 months'
        WHEN "Tenure in Months" <= 48 THEN '25-48 months'
        ELSE '49-72 months'
    END AS tenure_group,
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN "Customer Status" = 'Churned' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN "Customer Status" = 'Churned' THEN 1
                    ELSE 0
                END
            ) AS REAL
        ) * 100 / COUNT(*),
        2
    ) AS churn_percentage
    FROM customers
    GROUP BY tenure_group
    ORDER BY
        CASE tenure_group
            WHEN '0-12 months' THEN 1
            WHEN '13-24 months' THEN 2
            WHEN '25-48 months' THEN 3
            ELSE 4
        END;
    """,
    conn
)

,tenure_group,total_customers,churned_customers,churn_percentage
0,0-12 months,2186,1037,47.44
1,13-24 months,1024,294,28.71
2,25-48 months,1594,325,20.39
3,49-72 months,2239,213,9.51


### SQL Finding: Churn by Customer Tenure

The SQL analysis shows that observed churn percentages decrease across the four customer-tenure groups.

Customers with 0–12 months of tenure had the highest churn percentage at 47.44%, followed by customers with 13–24 months at 28.71%. Customers with 25–48 months had a churn percentage of 20.39%, while those with 49–72 months had the lowest percentage at 9.51%.

**Business implication:** The company should investigate the early customer experience, particularly during the first 12 months. Possible areas for further investigation include onboarding, service reliability, customer support, pricing, and whether customers' expectations are being met.

**Conclusion:** Customer tenure is strongly associated with observed churn in this dataset. Further analysis is needed to understand the factors contributing to this pattern.

In [28]:
# Churn by payment method

pd.read_sql_query(
    """
SELECT
    "Payment Method",
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN "Customer Status" = 'Churned' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN "Customer Status" = 'Churned' THEN 1
                    ELSE 0
                END
            ) AS REAL
        ) * 100 / COUNT(*),
        2
    ) AS churn_percentage
FROM customers
GROUP BY "Payment Method"
ORDER BY churn_percentage DESC;
""",
    conn
)

,Payment Method,total_customers,churned_customers,churn_percentage
0,Mailed Check,385,142,36.88
1,Bank Withdrawal,3909,1329,34.00
2,Credit Card,2749,398,14.48


### SQL Finding: Churn by Payment Method

The SQL analysis identified differences in observed churn percentages across payment methods. Customers paying by mailed check had the highest churn percentage at 36.88%, followed by customers using bank withdrawal at 34.00%. Customers paying by credit card had the lowest churn percentage at 14.48%.

Although mailed-check customers had the highest churn percentage, bank withdrawal customers accounted for the largest number of churned customers (1,329), reflecting the larger size of that group.

**Business implication:** The company should investigate whether billing experience, payment preferences, contract type, or other customer characteristics contribute to these differences. Any proposed billing or retention changes should be tested rather than assumed to reduce churn.

**Limitation:** These results show an association between payment method and customer status. They do not establish that payment method causes churn.

# SQL Analysis — Summary of Findings

## Objective

The objective of this analysis was to use SQL to investigate customer churn patterns across internet type, contract type, customer tenure, and payment method.

## Key Findings

1. **Internet Type:** Fiber Optic customers had the highest observed churn percentage (40.72%), followed by Cable customers (25.66%) and DSL customers (18.58%). This analysis included only customers with internet service.

2. **Contract Type:** Month-to-month customers had an observed churn percentage of 45.84%, compared with 10.71% for one-year customers and 2.55% for two-year customers.

3. **Customer Tenure:** Customers with 0–12 months of tenure had the highest observed churn percentage (47.44%). The percentage declined across the longer-tenure groups, reaching 9.51% among customers with 49–72 months of tenure.

4. **Payment Method:** Mailed-check customers had the highest observed churn percentage (36.88%), followed by bank withdrawal customers (34.00%). Credit card customers had the lowest percentage (14.48%).

## Business Implications

The results identify several groups that warrant further investigation, including month-to-month customers, customers in their first year, Fiber Optic customers, and customers using mailed checks or bank withdrawal.

The company should investigate possible explanations such as pricing, contract terms, service quality, billing experience, and customer support before implementing retention initiatives.

## Limitations

This analysis is descriptive. The observed relationships do not prove that any particular contract type, tenure group, internet type, or payment method causes churn. The factors may also be related to one another, so further analysis is needed to understand their combined effects.

## Conclusion

SQL provided a repeatable way to calculate customer counts and churn percentages across important customer segments. These results reinforce several patterns identified during exploratory data analysis and provide a foundation for deeper analysis, visualizations, and business recommendations.